# Notebook 04: Audit XAI, Concordanza By-Design vs. Post-Hoc & Risposta a RQ1/RQ2
## Fase 4 del Piano Operativo (`results/final_experimental_report.md`)

**Obiettivi Scientifici e Metodologici:**
1. **Risposta a RQ1 (Allineamento Inter-Modello - *XAI for Explanation*)**:
   - Valutare se a parità di performance (F1 $>99\%$), i tre modelli (`Lopez17CNN`, `BiflowTransformer`, `MedDefenderTransformer`) convergono sull'attribuzione degli stessi campi di rete ($F=6$) e pacchetti ($T=10$) o sviluppano logiche decisionali dissimili.
2. **Risposta a RQ2 (Audit By-Design vs. Post-Hoc - *Affidabilità a Costo Zero*)**:
   - **Concordanza di Ranking (Spearman $\rho \uparrow$)**: ordinamento dei 10 pacchetti temporali tra SHAP e Attenzione Intrinseca (*Attention Rollout* e *MHAN*).
   - **Allineamento Geometrico (Cosine Similarity $CS \uparrow$)**: allineamento vettoriale in $\mathbb{R}^{10}$.
   - **Verifica Causale sul Campo (Temporal Masking)**: mascheramento a inferenza dei top-$k$ pacchetti guidato dall'Attenzione by-design vs SHAP temporale per provare che l'attenzione ha la stessa fedeltà causale di SHAP.
   - **Costo Computazionale Edge**: divario temporale tra inferenza by-design (millisecondi, integrata nel forward pass) e SHAP post-hoc (secondi).
   - **Esportazione Tabella 3**: generazione di `results/metrics/xai_audit_table.csv` pronta per `results/final_experimental_report.md`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
from util.xai_utils import compute_temporal_alignment
from xai.model_loader import load_checkpoint

seed_everything(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso: {device}")

### 4.1 Caricamento Spiegazioni XAI e Metriche di Latenza Edge (Task T4.1)
Carichiamo l'archivio `results/metrics/xai_explanations.npz` (generato da Notebook 03) e i profili di latenza da `results/metrics/edge_latency_profiling.csv`.

In [ ]:
npz_path = METRICS_DIR / 'xai_explanations.npz'
data = np.load(npz_path, allow_pickle=True)  # generated by Notebook 03
shap_lopez_pkt = data['shap_lopez_pkt']
shap_biflow_pkt = data['shap_biflow_pkt']
shap_meddef_pkt = data['shap_meddef_pkt']

shap_lopez_field = data['shap_lopez_field']
shap_biflow_field = data['shap_biflow_field']
shap_meddef_field = data['shap_meddef_field']

attn_biflow_pkt = data['attn_biflow_pkt']
attn_meddef_pkt = data['attn_meddef_pkt']
field_names = list(data['field_names'])

# Explained flows from Notebook 03 (Temporal Masking on the same subset) and measured SHAP latency [ms/flow]
x_xai = torch.tensor(data['x_xai'])
y_xai = data['y_xai']
preds_xai = {'Lopez17CNN': data['pred_lopez'], 'BiflowTransformer': data['pred_biflow'],
             'MedDefenderTransformer': data['pred_meddef']}
# Real packets: semantic padding (torch.all(x == 0, dim=-1)) is excluded from alignment comparisons
valid_pkt = ~torch.all(x_xai.squeeze(1) == 0, dim=-1).numpy()  # [N, T]
shap_lat_ms = {'Lopez17CNN': float(data['shap_latency_lopez_ms']),
               'BiflowTransformer': float(data['shap_latency_biflow_ms']),
               'MedDefenderTransformer': float(data['shap_latency_meddef_ms'])}
print(f"[OK] Spiegazioni caricate con successo da {npz_path}")

# Single-thread edge CPU latency (Table 2, generated by scripts/profile_latency.py --all)
df_lat = pd.read_csv(METRICS_DIR / 'edge_latency_profiling.csv')
if df_lat['Checkpoint'].astype(str).str.startswith('None').any():
    raise RuntimeError("edge_latency_profiling.csv non e' misurato sui checkpoint addestrati: eseguire "
                       "'python scripts/profile_latency.py --all' dopo il Batch Job (Guida 14.3 regola 5).")
lat_map = dict(zip(df_lat['Network'], df_lat['Latency_Median_ms']))

### 4.2 Risposta a RQ1: Convergenza delle Strategie Decisionali tra i 3 Modelli
Verifichiamo se a livello di campo ($F=6$) e di profilo temporale ($T=10$) i 3 modelli guardano gli stessi elementi o divergono.

In [ ]:
# Median ranking per header field
med_lopez_f = np.median(np.abs(shap_lopez_field), axis=0)
med_biflow_f = np.median(np.abs(shap_biflow_field), axis=0)
med_meddef_f = np.median(np.abs(shap_meddef_field), axis=0)

df_rq1_fields = pd.DataFrame({
    'Campo': field_names,
    'Lopez17CNN_SHAP': med_lopez_f,
    'BiflowTransformer_SHAP': med_biflow_f,
    'MedDefenderTransformer_SHAP': med_meddef_f
})
df_rq1_fields['Rank_Lopez'] = df_rq1_fields['Lopez17CNN_SHAP'].rank(ascending=False).astype(int)
df_rq1_fields['Rank_Biflow'] = df_rq1_fields['BiflowTransformer_SHAP'].rank(ascending=False).astype(int)
df_rq1_fields['Rank_MedDefender'] = df_rq1_fields['MedDefenderTransformer_SHAP'].rank(ascending=False).astype(int)

# Inter-model Spearman rank correlation across header fields
rho_cnn_tr, _ = spearmanr(med_lopez_f, med_biflow_f)
rho_cnn_md, _ = spearmanr(med_lopez_f, med_meddef_f)
rho_tr_md, _ = spearmanr(med_biflow_f, med_meddef_f)

print("======================================================================")
print(" RISPOSTA A RQ1: ALLINEAMENTO INTER-MODELLO (CROSS-ARCHITECTURAL)")
print("======================================================================")
print(df_rq1_fields.to_string(index=False))
print(f"\nCorrelazione Ranking Campi (Spearman):")
print(f"  - Lopez17CNN vs. BiflowTransformer:      rho = {rho_cnn_tr:.3f}")
print(f"  - Lopez17CNN vs. MedDefenderTransformer:  rho = {rho_cnn_md:.3f}")
print(f"  - BiflowTransformer vs. MedDefender:      rho = {rho_tr_md:.3f}")
for col, net in [('Rank_Lopez', 'Lopez17CNN'), ('Rank_Biflow', 'BiflowTransformer'), ('Rank_MedDefender', 'MedDefenderTransformer')]:
    print(f"Campo dominante {net:24s}: {df_rq1_fields.loc[df_rq1_fields[col] == 1, 'Campo'].values[0]}")

# Per-class RQ1 across all 13 classes: median |phi| profiles by Field (F=6) and Packet (T=10)
# on correctly classified flows per model, including dominant field polarity (+ red / - blue)
shap_views = {'Lopez17CNN': (shap_lopez_field, shap_lopez_pkt),
              'BiflowTransformer': (shap_biflow_field, shap_biflow_pkt),
              'MedDefenderTransformer': (shap_meddef_field, shap_meddef_pkt)}
rq1_rows = []
for c in range(13):
    row = {'Classe': c}
    for net, (f_view, p_view) in shap_views.items():
        sel = (y_xai == c) & (preds_xai[net] == c)
        row[f'N_{net}'] = int(sel.sum())
        if sel.any():
            f_top = int(np.argmax(np.median(np.abs(f_view[sel]), axis=0)))
            row[f'Campo_top_{net}'] = field_names[f_top]
            # Signed median of phi~ on the dominant field: > 0 excitatory (red), < 0 inhibitory (blue)
            row[f'Phi_segno_{net}'] = round(float(np.median(f_view[sel, f_top])), 4)
            row[f'Pacchetto_top_{net}'] = int(np.argmax(np.median(np.abs(p_view[sel]), axis=0))) + 1
        else:
            row[f'Campo_top_{net}'] = row[f'Phi_segno_{net}'] = row[f'Pacchetto_top_{net}'] = None
    row['Campo_concorde'] = len({row[f'Campo_top_{n}'] for n in shap_views}) == 1
    row['Pacchetto_concorde'] = len({row[f'Pacchetto_top_{n}'] for n in shap_views}) == 1
    rq1_rows.append(row)
df_rq1_class = pd.DataFrame(rq1_rows)
df_rq1_class.to_csv(METRICS_DIR / 'rq1_per_class_fields.csv', index=False)
print("\nRQ1 per classe (campo e pacchetto dominanti per modello):")
print(df_rq1_class.to_string(index=False))
print(f"Classi con campo dominante concorde tra i 3 modelli: {int(df_rq1_class['Campo_concorde'].sum())}/13")
print(f"Classi con pacchetto dominante concorde tra i 3 modelli: {int(df_rq1_class['Pacchetto_concorde'].sum())}/13")

### 4.3 Risposta a RQ2: Concordanza Temporale tra Attenzione Intrinseca e SHAP
Calcoliamo Spearman $\rho$ e Cosine Similarity tra il vettore a 10 pacchetti di SHAP e l'attenzione intrinseca, sui soli pacchetti reali di ciascun flusso (il padding è escluso).

In [ ]:
align_biflow = compute_temporal_alignment(shap_biflow_pkt, attn_biflow_pkt, valid_pkt)
align_meddef = compute_temporal_alignment(shap_meddef_pkt, attn_meddef_pkt, valid_pkt)

print("======================================================================")
print(" RISPOSTA A RQ2: CONCORDANZA ATTENZIONE INTRINSECA VS SHAP")
print("======================================================================")
print(f"BiflowTransformer (Attention Rollout vs. SHAP):")
print(f"  - Spearman rho:        {align_biflow['spearman_rho_mean']:.3f} +/- {align_biflow['spearman_rho_std']:.3f}")
print(f"  - Cosine Similarity:   {align_biflow['cosine_similarity_mean']:.3f} +/- {align_biflow['cosine_similarity_std']:.3f}")
print(f"\nMedDefenderTransformer (MHAN Attention vs. SHAP):")
print(f"  - Spearman rho:        {align_meddef['spearman_rho_mean']:.3f} +/- {align_meddef['spearman_rho_std']:.3f}")
print(f"  - Cosine Similarity:   {align_meddef['cosine_similarity_mean']:.3f} +/- {align_meddef['cosine_similarity_std']:.3f}")

### 4.4 Risposta a RQ2: Verifica Causale via Temporal Masking su Modelli Congelati (Task T4.3)
Mascheriamo i Top-$k$ pacchetti ($k=1..5$) guidati dall'attenzione intrinseca vs SHAP temporale sui modelli congelati (imputando il valore neutro `0.5` sui pacchetti validi), sugli stessi $N$ flussi spiegati in Notebook 03: il ranking SHAP esiste solo per questi flussi. Se il calo $\Delta\text{Macro-F1}$ ricalca quello di SHAP, l'attenzione a costo zero è causalmente equivalente a SHAP.

In [ ]:
def mask_top_k_packets(x, ranking, k=3, mask_val=0.5):
    """
    Maschera i top-k pacchetti per ciascun flusso sostituendo i campi con mask_val.
    x: tensore [B, 1, T, F] oppure [B, T, F]
    ranking: array [B, T] con score di importanza per pacchetto
    """
    x_masked = x.clone()
    for i in range(x.size(0)):
        r = ranking[i] if ranking.ndim == 2 else ranking
        top_pkts = np.argsort(-r)[:k]
        for t in top_pkts:
            if x_masked.dim() == 4:
                if x_masked[i, 0, t].abs().sum() > 0:
                    x_masked[i, 0, t, :] = mask_val
            else:
                if x_masked[i, t].abs().sum() > 0:
                    x_masked[i, t, :] = mask_val
    return x_masked

RUN_IDS = {'Lopez17CNN': 'S2_R01_Lopez17CNN_multi_10pkts_seed0',
           'BiflowTransformer': 'S2_R02_BiflowTransformer_multi_10pkts_seed0',
           'MedDefenderTransformer': 'S2_R03_MedDefender_multi_10pkts_seed0'}
eval_models = {n: load_checkpoint(exp_name=f"cic-iomt_scratch_{e}", results_dir=str(RESULTS_DIR),
                                  network=n, device=str(device))[0] for n, e in RUN_IDS.items()}

def macro_f1(m, x, bs=256):
    preds = []
    with torch.no_grad():
        for i in range(0, len(x), bs):
            out = m(x[i:i + bs].to(device))
            out = out[0] if isinstance(out, (list, tuple)) else out
            preds.append(out.argmax(dim=-1).cpu().numpy())
    return f1_score(y_xai, np.concatenate(preds), average="macro") * 100

K_STEPS = [1, 2, 3, 4, 5]
# Temporal SHAP: magnitude |phi_t|, consistent with compute_temporal_alignment.
# Padded packets are excluded from top-k selection.
only_valid = lambda r: np.where(valid_pkt, r, -np.inf)
rankings = {
    'Lopez17CNN_SHAP': ('Lopez17CNN', only_valid(np.abs(shap_lopez_pkt))),
    'BiflowTransformer_SHAP': ('BiflowTransformer', only_valid(np.abs(shap_biflow_pkt))),
    'BiflowTransformer_Attention': ('BiflowTransformer', only_valid(attn_biflow_pkt)),
    'MedDefenderTransformer_SHAP': ('MedDefenderTransformer', only_valid(np.abs(shap_meddef_pkt))),
    'MedDefenderTransformer_MHAN': ('MedDefenderTransformer', only_valid(attn_meddef_pkt)),
}
base_f1 = {n: macro_f1(m, x_xai) for n, m in eval_models.items()}
# Delta Macro-F1 [%] for k = 1..5
temporal_masking_curves = {
    key: [macro_f1(eval_models[net], mask_top_k_packets(x_xai, rank, k=k)) - base_f1[net] for k in K_STEPS]
    for key, (net, rank) in rankings.items()
}
temporal_masking_results = {key: curve[K_STEPS.index(3)] for key, curve in temporal_masking_curves.items()}  # Top-3 (Table 3)

print(f"Baseline Macro-F1 sui {len(y_xai)} flussi spiegati: " + ", ".join(f"{n} {v:.1f}%" for n, v in base_f1.items()))
for k_res, v_res in temporal_masking_results.items():
    print(f"  - {k_res:32s}: Delta F1 (Top-3) = {v_res:.1f}%")


### 4.5 Esportazione Tabella 3 Finale di Audit (`results/metrics/xai_audit_table.csv`) (Task T4.4)
Costruiamo la tabella comparativa finale specificata in `results/final_experimental_report.md` (Tabella 3, Sez. 11).

In [ ]:
audit_table_records = [
    {
        'Modello': 'Lopez17CNN',
        'Metodo di Spiegazione': 'GradientSHAP (Post-Hoc)',
        'Spearman rho (vs SHAP)': 'N/A (Baseline)',
        'Cosine Sim (vs SHAP)': 'N/A (Baseline)',
        'Latenza per Flusso': f"{shap_lat_ms['Lopez17CNN'] / 1000:.2f} s",
        'Costo Computazionale Edge': 'Proibitivo (O(M * samples))',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['Lopez17CNN_SHAP']:.1f}%"
    },
    {
        'Modello': 'BiflowTransformer',
        'Metodo di Spiegazione': 'GradientSHAP (Post-Hoc)',
        'Spearman rho (vs SHAP)': '1.000',
        'Cosine Sim (vs SHAP)': '1.000',
        'Latenza per Flusso': f"{shap_lat_ms['BiflowTransformer'] / 1000:.2f} s",
        'Costo Computazionale Edge': 'Proibitivo (O(M * samples))',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['BiflowTransformer_SHAP']:.1f}%"
    },
    {
        'Modello': 'BiflowTransformer',
        'Metodo di Spiegazione': 'Attention Rollout (By-Design)',
        'Spearman rho (vs SHAP)': f"{align_biflow['spearman_rho_mean']:.3f}",
        'Cosine Sim (vs SHAP)': f"{align_biflow['cosine_similarity_mean']:.3f}",
        'Latenza per Flusso': f"{lat_map['BiflowTransformer']:.2f} ms",
        'Costo Computazionale Edge': 'Zero-cost (integrato nel forward)',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['BiflowTransformer_Attention']:.1f}%"
    },
    {
        'Modello': 'MedDefenderTransformer',
        'Metodo di Spiegazione': 'GradientSHAP (Post-Hoc)',
        'Spearman rho (vs SHAP)': '1.000',
        'Cosine Sim (vs SHAP)': '1.000',
        'Latenza per Flusso': f"{shap_lat_ms['MedDefenderTransformer'] / 1000:.2f} s",
        'Costo Computazionale Edge': 'Proibitivo (O(M * samples))',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['MedDefenderTransformer_SHAP']:.1f}%"
    },
    {
        'Modello': 'MedDefenderTransformer',
        'Metodo di Spiegazione': 'MHAN Attention (By-Design)',
        'Spearman rho (vs SHAP)': f"{align_meddef['spearman_rho_mean']:.3f}",
        'Cosine Sim (vs SHAP)': f"{align_meddef['cosine_similarity_mean']:.3f}",
        'Latenza per Flusso': f"{lat_map['MedDefenderTransformer']:.2f} ms",
        'Costo Computazionale Edge': 'Zero-cost (integrato nel forward)',
        'Temporal Masking (Top-3 Delta F1)': f"{temporal_masking_results['MedDefenderTransformer_MHAN']:.1f}%"
    }
]

df_audit_table = pd.DataFrame(audit_table_records)
out_audit_csv = METRICS_DIR / 'xai_audit_table.csv'
df_audit_table.to_csv(out_audit_csv, index=False)

print("\n======================================================================")
print(" TABELLA 3: AUDIT CONCORDANZA E ALLINEAMENTO TEMPORALE (RQ1 & RQ2)")
print("======================================================================")
display(df_audit_table) if 'display' in globals() else print(df_audit_table.to_string(index=False))
print(f"\n[SUCCESS] Tabella salvata in: {out_audit_csv}")

### 4.6 Generazione Figure di Audit XAI con Unità di Misura Esplicite (Task T4.4)
Generiamo le 4 visualizzazioni comparative ad alta risoluzione (300 DPI) per il paper scientifico, specificando **tassativamente le unità di misura su tutti gli assi cartesiani**:
1. **Allineamento Temporale ($t \in [1..10]$)**: profilo temporale percentuale di SHAP vs. Attention Rollout vs. MHAN Pooling.
2. **Trade-off Computazionale Edge**: confronto in scala logaritmica della latenza [ms/flusso].
3. **Validazione Causale (Temporal Masking)**: crollo prestazionale $\Delta\text{Macro-F1}$ [%] per $k=1..5$ pacchetti.
4. **Convergenza Inter-Modello sui Campi**: importanza mediana normalizzata $|\tilde{\phi}_f|$ [a.u.] tra CNN, Transformer e MedDefender.

In [ ]:
# Publication plot styling configuration
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.titlesize': 13
})

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Temporal Alignment (SHAP vs Attention Rollout vs MHAN)
ax1 = axes[0, 0]
t_steps = np.arange(1, 11)

# Per-packet shares over real packets only (padding excluded, matching compute_temporal_alignment)
share = lambda v: v.mean(axis=0) / v.mean(axis=0).sum() * 100
p_shap_biflow = share(np.abs(shap_biflow_pkt) * valid_pkt)
p_attn_biflow = share(attn_biflow_pkt)
p_shap_meddef = share(np.abs(shap_meddef_pkt) * valid_pkt)
p_attn_meddef = share(attn_meddef_pkt)
p_shap_lopez  = share(np.abs(shap_lopez_pkt) * valid_pkt)

ax1.plot(t_steps, p_shap_lopez, 'o--', color='#d95f02', label='Lopez17CNN SHAP (Baseline Post-Hoc)', linewidth=1.8)
ax1.plot(t_steps, p_shap_biflow, 's-', color='#7570b3', label='BiflowTransformer SHAP (Post-Hoc)', linewidth=2.0)
ax1.plot(t_steps, p_attn_biflow, '^--', color='#e7298a', label='BiflowTransformer Rollout (By-Design)', linewidth=2.0)
ax1.plot(t_steps, p_shap_meddef, 'D-', color='#1b9e77', label='MedDefender SHAP (Post-Hoc)', linewidth=2.0)
ax1.plot(t_steps, p_attn_meddef, 'v--', color='#66a61e', label='MedDefender MHAN (By-Design)', linewidth=2.0)

ax1.set_title('A) Concordanza Temporale: SHAP vs. Attenzione Intrinseca (RQ1 & RQ2)', fontweight='bold')
ax1.set_xlabel('Indice Sequenza Temporale Pacchetto, t [1..10]')
ax1.set_ylabel('Quota di Rilevanza Temporale Normalizzata [% quota per pacchetto]')
ax1.set_xticks(t_steps)
ax1.set_ylim(0, 45)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right')

# 2. Edge Computational Trade-off (Log Latency)
ax2 = axes[0, 1]
methods = ['Lopez17CNN\nGradientSHAP', 'Biflow\nGradientSHAP', 'MedDefender\nGradientSHAP', 'Biflow\nRollout (By-Design)', 'MedDefender\nMHAN (By-Design)']
latencies_ms = [shap_lat_ms['Lopez17CNN'], shap_lat_ms['BiflowTransformer'], shap_lat_ms['MedDefenderTransformer'],
                float(lat_map['BiflowTransformer']), float(lat_map['MedDefenderTransformer'])]
colors_lat = ['#e41a1c', '#e41a1c', '#e41a1c', '#377eb8', '#4daf4a']

bars = ax2.bar(methods, latencies_ms, color=colors_lat, width=0.55, edgecolor='black', linewidth=0.8)
ax2.set_yscale('log')
ax2.set_title('B) Divario di Efficienza Computazionale a Inferenza (Scala Log)', fontweight='bold')
ax2.set_xlabel('Metodo di Spiegazione XAI e Architettura')
ax2.set_ylabel('Latenza di Spiegazione per Flusso su CPU Gateway [ms]')
ax2.grid(True, which='both', linestyle='--', alpha=0.4, axis='y')
ax2.axhline(y=10.0, color='red', linestyle=':', linewidth=1.5, label='Soglia Limite Real-Time Edge (10 ms)')

for bar in bars:
    val = bar.get_height()
    txt = f'{val:.2f} ms' if val < 10 else f'{val/1000:.1f} s'
    ax2.annotate(txt, (bar.get_x() + bar.get_width() / 2, val), ha='center', va='bottom', xytext=(0, 4), textcoords='offset points', fontsize=9, fontweight='bold')
ax2.legend(loc='upper right')

# 3. Causal Validation via Temporal Masking (Top-k Packets)
ax3 = axes[1, 0]
k_steps = np.array(K_STEPS)
delta_shap_biflow = np.array(temporal_masking_curves['BiflowTransformer_SHAP'])
delta_attn_biflow = np.array(temporal_masking_curves['BiflowTransformer_Attention'])
delta_shap_meddef = np.array(temporal_masking_curves['MedDefenderTransformer_SHAP'])
delta_attn_meddef = np.array(temporal_masking_curves['MedDefenderTransformer_MHAN'])

ax3.plot(k_steps, delta_shap_biflow, 's-', color='#7570b3', linewidth=2.0, label='BiflowTransformer (Guidato da SHAP)')
ax3.plot(k_steps, delta_attn_biflow, '^--', color='#e7298a', linewidth=2.0, label='BiflowTransformer (Guidato da Rollout)')
ax3.plot(k_steps, delta_shap_meddef, 'D-', color='#1b9e77', linewidth=2.0, label='MedDefender (Guidato da SHAP)')
ax3.plot(k_steps, delta_attn_meddef, 'v--', color='#66a61e', linewidth=2.0, label='MedDefender (Guidato da MHAN)')

ax3.set_title('C) Validazione Causale: Temporal Masking su Modelli Congelati (RQ2)', fontweight='bold')
ax3.set_xlabel('Numero di Top-k Pacchetti Mascherati a Inferenza, k [1..5]')
ax3.set_ylabel(r'Calo Prestazionale $\Delta$ Macro-F1 [%]')
ax3.set_xticks(k_steps)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.legend(loc='lower left')

# 4. Inter-Model Convergence on the 6 Header Fields (RQ1)
ax4 = axes[1, 1]
f_indices = np.arange(len(field_names))
w = 0.25

med_lopez_f = np.median(np.abs(shap_lopez_field), axis=0)
med_biflow_f = np.median(np.abs(shap_biflow_field), axis=0)
med_meddef_f = np.median(np.abs(shap_meddef_field), axis=0)

ax4.bar(f_indices - w, med_lopez_f, width=w, color='#d95f02', label='Lopez17CNN', edgecolor='black', linewidth=0.6)
ax4.bar(f_indices, med_biflow_f, width=w, color='#7570b3', label='BiflowTransformer', edgecolor='black', linewidth=0.6)
ax4.bar(f_indices + w, med_meddef_f, width=w, color='#1b9e77', label='MedDefenderTransformer', edgecolor='black', linewidth=0.6)

ax4.set_title('D) Importanza Globale dei Campi Header tra le 3 Architetture (RQ1)', fontweight='bold')
ax4.set_xlabel(r'Campi di Intestazione di Protocollo, $f \in \{PL, IAT, DIR, WIN, TTL, FLG\}$')
ax4.set_ylabel(r'Importanza SHAP Mediana Normalizzata $|\tilde{\phi}_f|$ [Quota adimensionale (a.u.)]')
ax4.set_xticks(f_indices)
ax4.set_xticklabels(field_names)
ax4.grid(True, linestyle='--', alpha=0.5, axis='y')
ax4.legend(loc='upper left')

plt.tight_layout()
out_fig_audit_png = FIGURES_DIR / 'xai_audit_comprehensive_metrics.png'
out_fig_audit_pdf = FIGURES_DIR / 'xai_audit_comprehensive_metrics.pdf'
plt.savefig(out_fig_audit_png, dpi=300, bbox_inches='tight')
plt.savefig(out_fig_audit_pdf, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] Figure di audit XAI ad alta risoluzione salvate in:\n  - {out_fig_audit_png}\n  - {out_fig_audit_pdf}')